# Floriscan Phase A — 10-class species training

Train EfficientNet-B0 in **Google Colab** and write weights to **Google Drive**.
Do not download datasets onto a Windows C: drive. After training, copy `floriscan_species.pth` to `D:\\Projects\\Floriscan\\backend\\models\\`.

**Locked classes:** rose, tulip, lily (`Lilium` only), sunflower, carnation, peony, iris (bearded), daffodil, hibiscus, cherry_blossom (`Prunus serrulata`, not plum).

**Must match the Flask loader:** input 384, RGB, ImageNet mean/std.

In [ ]:
# Do not pin opencv 4.8 / numpy 1.26 — that breaks current Colab (NumPy 2 + Python 3.13).
# EfficientNet-B0 state_dict still loads in Flask even if Colab's timm is newer than 0.9.12.
!pip -q install -U timm
!python -c "import numpy, torch, timm; print('numpy', numpy.__version__); print('torch', torch.__version__); print('timm', timm.__version__)"

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import os
import random
import shutil

import cv2
import matplotlib.pyplot as plt
import numpy as np
import timm
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import Flowers102
from torchvision.datasets.utils import download_and_extract_archive

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

SPECIES = [
    "rose", "tulip", "lily", "sunflower", "carnation",
    "peony", "iris", "daffodil", "hibiscus", "cherry_blossom",
]

DRIVE_ROOT = Path("/content/drive/MyDrive/Floriscan")
DATA_ROOT = DRIVE_ROOT / "data" / "species"
MERGED = DATA_ROOT / "merged"
EXPORT = DRIVE_ROOT / "exports"
CACHE = Path("/content/floriscan_cache")

for folder in SPECIES:
    (MERGED / folder).mkdir(parents=True, exist_ok=True)
EXPORT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 384
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)
print("Drive data root:", DATA_ROOT)

## Merge public datasets on Drive

- TensorFlow `flower_photos`: roses, tulips, sunflowers (ignore daisy/dandelion).
- Oxford 102: rose, sunflower, carnation, daffodil, hibiscus, yellow/bearded iris, tiger lily + fire lily. Do **not** map siam tulip, water lily, lotus, sword lily, Peruvian lily, canna lily.
- Optional Kaggle [5-flower types](https://www.kaggle.com/datasets/kausthubkannan/5-flower-types-classification-dataset): **Lilly** and **Tulip** only. Ignore orchid and lotus.
- Peony and cherry blossom are thin in those sets. Drop extra photos into the Drive folders printed below (iNaturalist / GBIF / Kaggle search).

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def is_image(path: Path) -> bool:
    return path.suffix.lower() in IMAGE_EXTS


def copy_unique(src: Path, dest_dir: Path, prefix: str):
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest = dest_dir / f"{prefix}_{src.name}"
    if dest.exists():
        return False
    shutil.copy2(src, dest)
    return True


def copy_tree_mapped(src_dir: Path, dest_name: str, prefix: str):
    if not src_dir.is_dir():
        print("Missing", src_dir)
        return 0
    count = 0
    for path in src_dir.rglob("*"):
        if path.is_file() and is_image(path):
            count += int(copy_unique(path, MERGED / dest_name, prefix))
    return count

In [ ]:
TF_URL = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
tf_root = CACHE / "flower_photos"
if not tf_root.exists():
    download_and_extract_archive(TF_URL, download_root=str(CACHE), extract_root=str(CACHE))

tf_map = {"roses": "rose", "tulips": "tulip", "sunflowers": "sunflower"}
for src_name, dest_name in tf_map.items():
    n = copy_tree_mapped(tf_root / src_name, dest_name, "tf")
    print(f"TF {src_name} -> {dest_name}: {n}")

In [ ]:
OXFORD_MAP = {
    "rose": "rose",
    "sunflower": "sunflower",
    "carnation": "carnation",
    "daffodil": "daffodil",
    "hibiscus": "hibiscus",
    "yellow iris": "iris",
    "bearded iris": "iris",
    "tiger lily": "lily",
    "fire lily": "lily",
}

for split in ("train", "val", "test"):
    Flowers102(root=str(CACHE / "oxford102"), split=split, download=True)
print("Oxford 102 downloaded to", CACHE / "oxford102")

Oxford file lookup can differ by torchvision version. The next cell copies by class name using the dataset's own files when available, then falls back to walking `jpg/`.

In [ ]:
from torchvision.datasets import Flowers102 as F102

root = CACHE / "oxford102"
copied = {name: 0 for name in SPECIES}
for split in ("train", "val", "test"):
    ds = F102(root=str(root), split=split, download=False)
    class_names = list(ds.classes)
    for i in range(len(ds)):
        label_idx = int(ds._labels[i])
        label = class_names[label_idx]
        dest_name = OXFORD_MAP.get(label)
        if not dest_name:
            continue
        img_name = ds._image_files[i]
        path = Path(img_name) if os.path.isabs(str(img_name)) else root / "flowers-102" / "jpg" / Path(img_name).name
        if not path.exists():
            matches = list((root / "flowers-102").rglob(Path(img_name).name))
            if not matches:
                continue
            path = matches[0]
        copied[dest_name] += int(copy_unique(path, MERGED / dest_name, f"ox_{split}"))
print(copied)

In [ ]:
# Optional Kaggle 5-flower types (lily + tulip only).
# 1. Upload kaggle.json via the Colab files panel, or skip this cell.
# 2. Dataset: kausthubkannan/5-flower-types-classification-dataset

kaggle_dir = CACHE / "kaggle5"
if Path("/root/.kaggle/kaggle.json").exists() or Path("kaggle.json").exists():
    os.makedirs("/root/.kaggle", exist_ok=True)
    if Path("kaggle.json").exists():
        shutil.copy("kaggle.json", "/root/.kaggle/kaggle.json")
    os.chmod("/root/.kaggle/kaggle.json", 0o600)
    !pip -q install kaggle
    !kaggle datasets download -d kausthubkannan/5-flower-types-classification-dataset -p {kaggle_dir} --unzip
    mapping = {"Lilly": "lily", "Tulip": "tulip", "lily": "lily", "tulip": "tulip"}
    for folder in kaggle_dir.rglob("*"):
        if folder.is_dir() and folder.name in mapping:
            n = copy_tree_mapped(folder, mapping[folder.name], "kg5")
            print(folder.name, n)
else:
    print("No kaggle.json — skip Kaggle merge. Lily/tulip still come from TF + Oxford.")

In [ ]:
print("Images per class in", MERGED)
counts = {}
for name in SPECIES:
    counts[name] = len([p for p in (MERGED / name).iterdir() if p.is_file() and is_image(p)])
    print(f"  {name:16s} {counts[name]}")

thin = [name for name, n in counts.items() if n < 80]
if thin:
    print("\nAdd more photos on Drive for:", ", ".join(thin))
    print("Peony / cherry blossom: search iNaturalist, GBIF, or Kaggle, then copy into:")
    for name in thin:
        print(" ", MERGED / name)

## Train EfficientNet-B0

70 / 15 / 15 split, light augmentation on train only, freeze then fine-tune.

In [ ]:
class FlowerFolder(Dataset):
    def __init__(self, items, train=False):
        self.items = items
        self.train = train

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        path, label = self.items[index]
        img = cv2.imread(str(path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
        if self.train:
            if random.random() < 0.5:
                img = cv2.flip(img, 1)
            if random.random() < 0.4:
                angle = random.uniform(-18, 18)
                matrix = cv2.getRotationMatrix2D((IMAGE_SIZE / 2, IMAGE_SIZE / 2), angle, 1.0)
                img = cv2.warpAffine(img, matrix, (IMAGE_SIZE, IMAGE_SIZE), borderMode=cv2.BORDER_REFLECT)
            hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV).astype(np.float32)
            hsv[:, :, 2] *= random.uniform(0.85, 1.15)
            hsv[:, :, 2] = np.clip(hsv[:, :, 2], 0, 255)
            img = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)
        img = img.astype(np.float32) / 255.0
        img = (img - MEAN) / STD
        tensor = torch.from_numpy(img).permute(2, 0, 1)
        return tensor, label


train_items, val_items, test_items = [], [], []
print("per-class 70/15/15 split (avoids empty test classes):")
for class_id, name in enumerate(SPECIES):
    files = [p for p in (MERGED / name).iterdir() if p.is_file() and is_image(p)]
    files.sort()
    random.shuffle(files)
    n = len(files)
    n_train = int(0.70 * n)
    n_val = int(0.15 * n)
    train_files = files[:n_train]
    val_files = files[n_train:n_train + n_val]
    test_files = files[n_train + n_val:]
    train_items.extend((path, class_id) for path in train_files)
    val_items.extend((path, class_id) for path in val_files)
    test_items.extend((path, class_id) for path in test_files)
    print(f"  {name:16s} n={n:4d}  train {len(train_files):4d}  val {len(val_files):4d}  test {len(test_files):4d}")

random.shuffle(train_items)
print("totals", len(train_items), len(val_items), len(test_items))

train_loader = DataLoader(FlowerFolder(train_items, train=True), batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(FlowerFolder(val_items, train=False), batch_size=16, shuffle=False, num_workers=2)
test_loader = DataLoader(FlowerFolder(test_items, train=False), batch_size=16, shuffle=False, num_workers=2)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device", device)

model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=len(SPECIES))
model.to(device)

for name, param in model.named_parameters():
    param.requires_grad = name.startswith("classifier")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)


def run_epoch(loader, train=True):
    model.train(train)
    total_loss = 0.0
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        if train:
            optimizer.zero_grad()
        with torch.set_grad_enabled(train):
            logits = model(images)
            loss = criterion(logits, labels)
            if train:
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / max(total, 1), correct / max(total, 1)


best_val = 0.0
best_path = EXPORT / "floriscan_species.pth"
for epoch in range(8):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    print(f"head {epoch+1:02d}  train {train_acc:.3f}  val {val_acc:.3f}  val_loss {val_loss:.4f}")
    if val_acc >= best_val:
        best_val = val_acc
        torch.save(model.state_dict(), best_path)

for param in model.parameters():
    param.requires_grad = True
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

for epoch in range(6):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    print(f"ft   {epoch+1:02d}  train {train_acc:.3f}  val {val_acc:.3f}  val_loss {val_loss:.4f}")
    if val_acc >= best_val:
        best_val = val_acc
        torch.save(model.state_dict(), best_path)

print("best val", best_val, "saved", best_path)

In [ ]:
model.load_state_dict(torch.load(best_path, map_location=device))
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for images, labels in test_loader:
        logits = model(images.to(device))
        y_true.extend(labels.tolist())
        y_pred.extend(logits.argmax(1).cpu().tolist())

label_ids = list(range(len(SPECIES)))
print("test images per class:")
for class_id, name in enumerate(SPECIES):
    n = y_true.count(class_id)
    print(f"  {name:16s} {n}")

print(
    classification_report(
        y_true,
        y_pred,
        labels=label_ids,
        target_names=SPECIES,
        digits=3,
        zero_division=0,
    )
)
matrix = confusion_matrix(y_true, y_pred, labels=label_ids)
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(matrix, cmap="Greens")
ax.set_xticks(range(len(SPECIES)), SPECIES, rotation=45, ha="right")
ax.set_yticks(range(len(SPECIES)), SPECIES)
ax.set_title("Floriscan species confusion matrix")
fig.colorbar(im)
plt.tight_layout()
fig.savefig(EXPORT / "species_confusion_matrix.png", dpi=140)
plt.show()
print("Copy this file to D:\\Projects\\Floriscan\\backend\\models\\floriscan_species.pth")
print(best_path)